# 1.2 Depth profile of raw survey recordings

**[env: optonpx]**

Each survey recording holds one shank, recorded with a sparse IMRO that spreads the 384 channels
over 0-8.6 mm of the shank (one recording per shank, because of the channel-map limits). When a recording holds several
probe streams, `PROBE_NAME` selects the one to analyse; caches and figures carry the probe name. From a
time window of the raw data (`duration_s`, default 60 s) this notebook computes per-channel depth
profiles, plots one figure per shank and an overview across shanks, to find the brain surface and
the region boundaries along the track. Raw data are read-only (`np.memmap` mode `"r"`); results
are cached as `.npz` in `output_dir` and reloaded on later runs (delete a `.npz` to recompute it;
a cache made with different parameters stops the notebook). Computation: `src/depth_profile.py`.

**Depth.** `depth below brain surface = implant_depth_um - tip_length_um - ypos`, where `ypos` is
the electrode height above the lowest electrode row and the implant depth is measured from the
brain surface to the probe tip. Depth 0 (black line) is the expected brain surface; channels
above it should be outside the brain.

**Panels of the per-shank figure**

1. **AP-band noise**: MAD/0.6745 of the 300-6000 Hz signal, no CAR. Robust estimate of the
   background noise (thermal + unresolved spikes). Flat channels (constant value) are marked x.
2. **Threshold crossings**: negative peaks of the CAR-referenced AP signal below
   `-threshold_mad` x MAD, at least `refractory_ms` apart, per second. Grey = channels, red =
   running median over depth. Mostly multi-unit activity: high in cell-dense grey matter, low in
   white matter, ventricles and outside the brain.
3. **Raster**: every crossing (time x depth, colour = amplitude). Shows layering, stability and
   drift. Time bins in which more than `artifact_fraction` of the channels cross together are
   global artifacts (movement / electrical); they are shaded red and excluded from panels 2-3.
4. **LFP band power**: Welch spectrum of the low-passed, decimated signal (2.5 kHz), mean power in
   each band, averaged per electrode row.
5. **Relative LFP spectrum**: each row's spectrum (dB) minus the median over depth at each
   frequency, so depth changes are visible despite the 1/f slope. Red = more power than the
   typical depth at that frequency.
6. **LFP correlation between depths, no CAR** (1-100 Hz): Pearson r between electrode rows.
   Dominated by the shared reference signal; tissue in the same volume conductor forms one
   block, and the brain surface usually shows up as a block edge.
7. **LFP correlation, CAR**: the median across channels is removed first, so only local
   differences remain; nearby rows correlate (diagonal band), and region / white-matter
   boundaries show as edges or kinks of the band.

The overview heatmaps place each shank at its AP coordinate (shank 0 posterior) and bin the
crossing rate and LFP band powers in depth. The probe maps (last cell) draw each metric on
the probe at true scale, every channel at its real position, for overlay on a sagittal atlas
section (the shanks lie along A-P).

In [ ]:
from pathlib import Path

# ========== Specify the survey ==========
survey_dir = Path(r"E:\D1-1-7_IM-1994\ephys_raw")
recordings = [                       # one recording per shank
    "2026-11-09_10-02-42",
    "2026-11-09_10-08-45",
    "2026-11-09_10-14-12",
    "2026-11-09_10-19-44",
]
output_dir = Path(r"E:\D1-1-7_IM-1994\2026-11-09_depth_profile")

implant_depth_um = 7300     # brain surface to probe tip
tip_length_um = 175         # tip to the lowest electrode row (NP2.0 user manual)
ap_center_mm = 1.7          # AP of the probe centre (ML 1.8); shanks along A-P
shank_pitch_mm = 0.25
shank0_posterior = True
PROBE_NAME = "ProbeB"       # probe stream to analyse, matched case-insensitively to the end of the
                            # stream name ("ProbeA", "ProbeB"); None = recordings with one probe only

t_start_s = 0.0
duration_s = 300.0          # analysed window per recording (clipped to the recording length)
# =========================================

In [ ]:
import json
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np

from src.depth_profile import (
    DEFAULT_PARAMS,
    channel_metrics,
    compute_depth_profile,
    load_depth_profile,
    plot_depth_profile,
    plot_probe_metric,
    plot_shank_overview,
    save_depth_profile,
)

output_dir.mkdir(parents=True, exist_ok=True)
params = {**DEFAULT_PARAMS, "t_start_s": t_start_s, "duration_s": duration_s}
direction = 1 if shank0_posterior else -1   # AP increases with shank index if shank 0 is posterior
shank_ap_mm = {s: ap_center_mm + direction * (s - 1.5) * shank_pitch_mm for s in range(4)}
probe_tag = f"_{PROBE_NAME}" if PROBE_NAME else ""   # in cache / figure names
probe_title = f"  {PROBE_NAME}" if PROBE_NAME else ""
print("shank AP (mm):", {s: round(v, 3) for s, v in shank_ap_mm.items()})

In [ ]:
# --- compute (or load the cached) depth profile of each recording ---
profiles = []
for name in recordings:
    cache = output_dir / f"{name}{probe_tag}_t{t_start_s:g}-{duration_s:g}s.npz"
    if cache.exists():
        profile = load_depth_profile(cache)
        # round-trip through JSON so tuples / lists compare equal
        if json.dumps(profile["params"], sort_keys=True) != json.dumps(json.loads(json.dumps(params)), sort_keys=True):
            raise ValueError(f"{cache} was made with different parameters -- delete it to recompute")
        print(f"{name}: loaded {cache.name}")
    else:
        print(f"{name}: computing ...", flush=True)
        profile = compute_depth_profile(survey_dir / name, probe_name=PROBE_NAME,
                                        t_start_s=t_start_s, duration_s=duration_s)
        save_depth_profile(profile, cache)
    print(f"  {profile.get('probe', '')} shank {np.unique(profile['shank']).tolist()}, window {profile['duration_s']:.1f} s "
          f"(clean {profile['clean_duration_s']:.1f} s), {len(profile['peak_time_s'])} crossings, "
          f"flat channels {np.flatnonzero(profile['flat']).tolist()}, "
          f"artifact bins {len(profile['artifact_bin_start_s'])}")
    profiles.append(profile)

In [ ]:
# --- one figure per shank ---
for profile in profiles:
    shank = int(profile["shank"][0])
    fig = plot_depth_profile(profile, implant_depth_um, tip_length_um,
                             title=f"{profile['recording']}{probe_title}  shank {shank}  (AP {shank_ap_mm[shank]:.3f} mm)")
    fig.savefig(output_dir / f"depth_profile{probe_tag}_shank{shank}_{profile['recording']}.png", dpi=300)
    plt.show()

In [ ]:
# --- overview across shanks ---
fig = plot_shank_overview(profiles, shank_ap_mm, implant_depth_um, tip_length_um)
fig.savefig(output_dir / f"depth_profile{probe_tag}_overview.png", dpi=300)
plt.show()

In [ ]:
# --- one figure per metric, drawn on the probe at true scale, for overlay on a sagittal atlas section ---
# Every channel is a 12 x 12 um square at its real position; x = AP (anterior left), y = depth below
# the brain surface; 1 mm horizontal = 1 mm vertical. Saved as png (300 dpi) and svg (vector, for
# scaling onto the atlas). One colour scale over all shanks (2nd-98th percentile).
probe_map_dir = output_dir / f"probe_maps{probe_tag}_t{t_start_s:g}-{duration_s:g}s"
probe_map_dir.mkdir(exist_ok=True)
for metric in channel_metrics(profiles[0]):
    fig = plot_probe_metric(profiles, metric, shank_ap_mm, implant_depth_um, tip_length_um)
    for ext in ("png", "svg"):
        fig.savefig(probe_map_dir / f"probe_{metric}.{ext}", dpi=300)
    plt.show()
print(f"saved to {probe_map_dir}")